# Sensores IoT: Ocupação das Salas

Em um complexo tecnológico e industrial, que está equipado com sensores ambientais IoT, se faz necessário prever em tempo real se salas técnicas e escritórios estão ocupados ou não, a fim de automatizar sistemas inteligentes de iluminação e ar-condicionado (HVAC), visando redução de custos com energia.

A variável alvo do projeto é binária: `Occupancy` = 1 (ambiente ocupado / ativo) e `Occupancy` = 0 (ambiente desocupado / inativo).

**Base de Dados**: Occupancy Detection Dataset (UCI Machine Learning Repository / Kaggle), composta pelas seguintes variáveis ambientais:
- `Temperature`: Temperatura do ambiente em graus Celsius (°C).
- `Humidity`: Umidade relativa do ar (%).
- `Light`: Nível de luminosidade medido pelo sensor (Lux).
- `CO2`: Concentração de dióxido de carbono no ar (ppm).
- `HumidityRatio`: Taxa derivada de umidade específica e temperatura.
- `Occupancy` (Variável Alvo): Indicador binário de presença (0 = Desocupado, 1 = Ocupado).


**Objetivo do Projeto:**

Comparar KNN e Árvore de Decisão e escolher o modelo para a automação predial.


**Fases do Projeto:**
- Fase 1: Análise Exploratória (EDA)
- Fase 2: Limpeza e Tratamento de Dados (Data Prep)
- Fase 3: Feature Engineering
- Fase 4: Divisão e Balanceamento dos Dados
- Fase 5: Escalonamento de Variáveis (StandardScaler)
- Fase 6: Ajuste de Parâmetros e Combate ao Overfitting
- Fase 7: Avaliação da Acurácia e Veredito Final



### Importação das bibliotecas

Importar as bibliotecas utilizadas em todo o projeto:

- **pandas** e **numpy**: leitura e manipulação dos dados.
- **matplotlib** e **seaborn**: criação dos gráficos da análise exploratória.
- **scikit-learn** (Machine Learning):
  - `train_test_split`: divisão da base em treino e teste;
  - `StandardScaler`: padronização da escala das variáveis para o KNN;
  - `KNeighborsClassifier` e `DecisionTreeClassifier`: os dois modelos comparados;
  - `accuracy_score`, `classification_report` e `confusion_matrix`: avaliação dos modelos.
- **os**, **zipfile** e **urllib.request**: download e extração da base de dados.

Também são definidas as configurações visuais padrão dos gráficos (fundo sem grade e tamanho da figura).

In [4]:
# Bibliotecas
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Configurações visuais para os gráficos
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (10, 5)

## Fase 1: Análise Exploratória (EDA)

Antes de treinar qualquer modelo, é preciso conhecer a base: quantas linhas e colunas existem, quais são os tipos de dados, como os valores se distribuem e como as variáveis se relacionam com a ocupação da sala. Essa etapa orienta todas as decisões das fases seguintes.

### Carregamento da base de dados

**Fonte:** Occupancy Detection Dataset, disponível no UCI Machine Learning Repository.

A base é baixada diretamente do repositório da UCI, em vez da versão disponibilizada em planilha online. A versão em planilha apresentou problemas de formatação: ao ser importada com configuração regional em português, o ponto decimal foi interpretado como separador de milhar, e valores como `0.0047` em `HumidityRatio` passaram a aparecer como números inteiros muito grandes. O arquivo original da UCI não passa por nenhuma planilha e preserva os valores corretos.

O arquivo baixado contém três partes da mesma coleta (`datatraining.txt`, `datatest.txt` e `datatest2.txt`). As três são lidas e unidas em um único DataFrame, formando a base completa, que é salva em `data/occupancy_completo.csv`.

O download acontece apenas na primeira execução: se o arquivo já existir na pasta `data/`, o código pula direto para a leitura. Todos os caminhos são relativos à pasta do notebook, para que o projeto rode em qualquer computador.

In [7]:
caminho_csv = "../data/occupancy_completo.csv"

# Baixar e extrair a base da UCI apenas se o arquivo final ainda não existir
if not os.path.exists(caminho_csv):
    url_uci = "https://archive.ics.uci.edu/ml/machine-learning-databases/00357/occupancy_data.zip"
    zip_path = "../data/occupancy_data.zip"
    urllib.request.urlretrieve(url_uci, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("../data/raw")

    # Ler as três partes da base
    df_treino = pd.read_csv("../data/raw/datatraining.txt")
    df_teste1 = pd.read_csv("../data/raw/datatest.txt")
    df_teste2 = pd.read_csv("../data/raw/datatest2.txt")

    # Unir as três partes em um único DataFrame
    df_completo = pd.concat([df_treino, df_teste1, df_teste2], ignore_index=True)

    # Salvar a base completa
    df_completo.to_csv(caminho_csv, index=False)

# Carregar a base completa
df = pd.read_csv(caminho_csv)
print("Base de dados carregada com sucesso!")

Base de dados carregada com sucesso!


In [8]:
df.head(10)

,date,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
0,2015-02-04 17:51:00,23.180,27.2720,426.0,721.250000,0.004793,1
1,2015-02-04 17:51:59,23.150,27.2675,429.5,714.000000,0.004783,1
2,2015-02-04 17:53:00,23.150,27.2450,426.0,713.500000,0.004779,1
3,2015-02-04 17:54:00,23.150,27.2000,426.0,708.250000,0.004772,1
4,2015-02-04 17:55:00,23.100,27.2000,426.0,704.500000,0.004757,1
5,2015-02-04 17:55:59,23.100,27.2000,419.0,701.000000,0.004757,1
6,2015-02-04 17:57:00,23.100,27.2000,419.0,701.666667,0.004757,1
7,2015-02-04 17:57:59,23.100,27.2000,419.0,699.000000,0.004757,1
8,2015-02-04 17:58:59,23.100,27.2000,419.0,689.333333,0.004757,1
9,2015-02-04 18:00:00,23.075,27.1750,419.0,688.000000,0.004745,1


As primeiras linhas confirmam que a base foi carregada corretamente, com sete colunas: a data e hora da medição (`date`), as cinco variáveis dos sensores (`Temperature`, `Humidity`, `Light`, `CO2` e `HumidityRatio`) e a variável alvo `Occupancy`.

Os valores numéricos aparecem com ponto decimal e na escala esperada (por exemplo, `HumidityRatio` com valores próximos de 0,005), sem a distorção observada na versão em planilha.

Os arquivos `.txt` da UCI trazem, no início de cada linha, um número de registro sem nome no cabeçalho. Ao ler os arquivos, o pandas interpretou essa coluna como índice, e não como dado. Na união das três partes, o parâmetro `ignore_index=True` renumerou as linhas em uma sequência única, e o `index=False` impediu que esse índice fosse gravado no arquivo final. Por isso, a base final contém apenas as colunas de interesse.

### Dimensões da base

Verificar quantas linhas (medições) e quantas colunas (variáveis) a base possui. O tamanho da base ajuda a avaliar se há dados suficientes para treinar e testar os modelos.

In [10]:
df.shape

(20560, 7)

### Tipos de dados e valores não nulos

Verificar o tipo de cada coluna e a quantidade de valores preenchidos. Os modelos de Machine Learning trabalham apenas com números, então é preciso confirmar que as variáveis dos sensores foram lidas como numéricas e identificar colunas que exigem algum tratamento antes da modelagem. A contagem de valores não nulos também dá uma primeira indicação sobre a existência de dados ausentes, que serão tratados na Fase 2.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20560 entries, 0 to 20559
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           20560 non-null  str    
 1   Temperature    20560 non-null  float64
 2   Humidity       20560 non-null  float64
 3   Light          20560 non-null  float64
 4   CO2            20560 non-null  float64
 5   HumidityRatio  20560 non-null  float64
 6   Occupancy      20560 non-null  int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 1.1 MB


### Resumo estatístico

Calcular as principais medidas estatísticas das colunas numéricas: contagem, média, desvio padrão, valores mínimo e máximo e quartis (25%, 50% e 75%). Esse resumo mostra a escala de cada sensor, ajuda a identificar possíveis valores extremos e permite comparar média e mediana (50%) para avaliar a simetria das distribuições.

In [12]:
df.describe()

,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
count,20560.000000,20560.000000,20560.000000,20560.000000,20560.000000,20560.000000
mean,20.906212,27.655925,130.756622,690.553276,0.004228,0.231031
std,1.055315,4.982154,210.430875,311.201281,0.000768,0.421503
min,19.000000,16.745000,0.000000,412.750000,0.002674,0.000000
25%,20.200000,24.500000,0.000000,460.000000,0.003719,0.000000
50%,20.700000,27.290000,0.000000,565.416667,0.004292,0.000000
75%,21.525000,31.290000,301.000000,804.666667,0.004832,0.000000
max,24.408333,39.500000,1697.250000,2076.500000,0.006476,1.000000
